<a href="https://colab.research.google.com/github/nabinjoshi54/lis5693/blob/main/lab-8/lab_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lab 8: Topic Modeling Using BERTopic

In this lab, I use the BERTopic package to perform topic modeling on the same dataset that I used in Lab 5. The goal is to explore how BERTopic groups documents into topics and then compare those results with the LDA-based topic modeling from the previous lab. Since BERTopic uses embeddings and clustering instead of only word-frequency patterns, it can capture semantic meaning in a different way.
In Google Colab, it is better to use a GPU for BERTopic because generating embeddings can take more time on CPU.

To enable GPU in Colab:

Runtime > Change runtime type > Hardware accelerator > GPU

Step 1: Install the required packages

First, I install the libraries needed for this lab. BERTopic depends on sentence transformers for embeddings and UMAP/HDBSCAN for clustering.

In [ ]:
%%capture
!pip install bertopic sentence-transformers umap-learn hdbscan plotly

Step 2: Import libraries
I import the Python libraries that will be used for loading the dataset, cleaning the text, building the BERTopic model, and visualizing the results.

In [ ]:
import pandas as pd
import numpy as np
import re

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

Step 3: Load the dataset

I use the same batteries-and-space dataset from Lab 5. This keeps the comparison fair because both LDA and BERTopic are being applied to the same collection of documents.

In [ ]:
url = "https://raw.githubusercontent.com/nabinjoshi54/lis5693/main/lab-8/lab5-batteries-and-space.csv"
df = pd.read_csv(url)

df.head()

Step 4: Inspect the data

Before modeling, I check the shape of the dataset and confirm that the abstract column is available because BERTopic will use document text as input.

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Step 5: Prepare the text data

For topic modeling, I use the `Abstract` column because it contains the main content of each article. I remove missing values, convert everything to string, and do light cleaning so the text is more consistent.

In [ ]:
df = df.dropna(subset=["Abstract"]).copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)          # remove extra spaces
    text = re.sub(r"\n", " ", text)           # remove line breaks
    return text.strip()

df["clean_abstract"] = df["Abstract"].apply(clean_text)

documents = df["clean_abstract"].tolist()

print("Number of documents used:", len(documents))
print("\nSample abstract:\n")
print(documents[0][:1000])

Step 6: Create document embeddings

BERTopic works differently from LDA because it starts by converting each document into an embedding. These embeddings capture semantic meaning, which helps BERTopic group documents based on similarity in meaning rather than only matching word frequencies.

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Step 7: Build the BERTopic model

Here, I create the BERTopic model. I also define a vectorizer with English stop words removed so the topic words are easier to interpret.

In [ ]:
vectorizer_model = CountVectorizer(stop_words="english")

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="english",
    calculate_probabilities=True,
    verbose=True
)

Step 8: Fit the model to the abstracts

Now I run BERTopic on the dataset. The model will generate topic assignments for each document and estimate topic probabilities.

In [ ]:
topics, probabilities = topic_model.fit_transform(documents)

Step 9: Review the discovered topics

After fitting the model, I inspect the topic summary table. This helps me see how many topics were created, how many documents are assigned to each topic, and which words best represent each topic.

In [ ]:
topic_info = topic_model.get_topic_info()
topic_info.head(15)

Step 10: Display the top words for several topics

To better understand the model output, I print the representative words from the first few discovered topics.

In [ ]:
valid_topics = [t for t in topic_info["Topic"].tolist() if t != -1][:10]

for topic_id in valid_topics:
    print(f"\nTopic {topic_id}:")
    print(topic_model.get_topic(topic_id))

Step 11: Add topic labels back to the dataset

To make the results easier to inspect, I add the assigned topic number to the original dataframe.

In [ ]:
df["BERTopic_Topic"] = topics
df[["Title", "BERTopic_Topic"]].head(10)

Step 12: Examine a few example documents from selected topics

Looking at example article titles helps confirm whether the topics make sense and whether similar articles are grouped together.

In [ ]:
for topic_id in valid_topics[:5]:
    print(f"\n{'='*80}")
    print(f"Examples from Topic {topic_id}")
    print(f"{'='*80}")

    sample_titles = df[df["BERTopic_Topic"] == topic_id]["Title"].head(5).tolist()
    for i, title in enumerate(sample_titles, 1):
        print(f"{i}. {title}")

Step 13: Visualize the topic map

This visualization shows how the topics are positioned relative to one another in a reduced-dimensional space. Topics that appear closer together are generally more semantically related.

In [ ]:
topic_model.visualize_topics()

Step 14: Visualize topic frequency

This chart shows which topics appear most often in the dataset.

In [ ]:
topic_model.visualize_barchart(top_n_topics=10)

Step 15: Visualize the hierarchy of topics

This figure helps show whether some topics can be grouped into broader higher-level themes.

In [ ]:
topic_model.visualize_hierarchy()

Task 2: Interpretation and Comparison with Lab 5

In Lab 5, I used LDA and selected the 20-topic model because it had the highest coherence score (0.41). The model identified key themes such as batteries, space systems, thermal behavior, and energy applications. However, many topics had overlapping keywords like "battery", "space", and "power", which made them harder to distinguish.

In Lab 8, BERTopic produced more distinct and meaningful topics. For example, Topic 7 clearly represents battery discharge (battery, lithium, discharge, ion), Topic 6 focuses on spacecraft systems (spacecraft, orbit, launched), and Topic 3 captures thermal behavior (thermal, temperature, heat). These topics are more focused compared to LDA.

The visualizations also show better separation. The topic map shows clearer clustering, and the hierarchical clustering groups related topics such as space systems and battery technologies together, while still keeping them distinct.

Both methods identified similar themes, but BERTopic produced more coherent and well-separated topics, while LDA showed more overlap between related concepts.

Task 3: Reflection

This lab helped me better understand how different topic modeling approaches work on the same dataset. One thing that went well was that both LDA and BERTopic were able to identify the main themes related to batteries and space applications. BERTopic produced more clear and meaningful topics and made interpretation easier.

One challenge I faced was understanding why LDA topics overlapped so much. The coherence score was higher for the 20-topic model butsome topics were still difficult to distinguish. This showed me that a higher coherence score does not always mean better interpretability. BERTopic also required more time to run because it uses embeddings, which can be computationally expensive.

In my future research, I think embeddings and large language models can be very useful. I found one article published by professor from UC Berkley that did the same work. I will probably do the same work for my research. This can help group similar research papers, identify important trends, and make literature reviews more efficient. I would like to apply these methods to analyze papers in my own research area.